In [1]:
import json
import os
import time
from pathlib import Path

import lseg.data as ld
from lseg.data.content import historical_pricing
import numpy as np
import pandas as pd
import yfinance as yf

from finder_core import load_settings

In [2]:
settings = load_settings(display=True, namespace=globals())
analysis_start_date = settings["analysis_start_date"]
end_date = settings["analysis_end_date"]
window_size = int(settings["window_size"])
download_start_date = (pd.Timestamp(analysis_start_date) - pd.offsets.BDay(max(200, 2 * window_size, 60) + window_size + 31)).strftime("%Y-%m-%d")
max_interpolation_days = 3

refinitiv_assets = {"^GSPC": ".SPX", "SPGTCLTR": ".SPGTCLTR", "STOXX": ".STOXX", "CACT": ".CACT", "GDAXI": ".GDAXI", "CBU0.L": "CBU0.L", "FTSE100": ".FTSE", "DJCI": ".DJCI"}
yahoo_assets = {"^GSPC": "^GSPC", "SPGTCLTR": None, "STOXX": "^STOXX", "CACT": "^SBF250", "GDAXI": "^GDAXI", "CBU0.L": "CBU0.L", "FTSE100": "^FTSE", "DJCI": "^DJCI"}
other_source_files = {}  # Optional: asset -> verified same-instrument CSV with Date and OHLCV columns.
refinitiv_fields = {"TRDPRC_1": "Close", "HIGH_1": "High", "LOW_1": "Low", "OPEN_PRC": "Open", "ACVOL_UNS": "Volume"}
columns = list(refinitiv_fields.values())
print(f"Analysis: {analysis_start_date} to {end_date} | Download including warm-up: {download_start_date}")

# Data
  TICKER = '^GSPC'
  START_DATE = '2024-09-01'
  END_DATE = '2026-08-31'
  NEWS_START_DATE = '2024-09-01'
  NEWS_END_DATE = '2026-08-31'
  WINDOW_SIZE = 126
  HORIZON_DAYS = 1
  CRASH_PERCENTILE = 0.05   # 0.01, 0.05
  RANDOM_SEED = 42
  INDICATOR_COUNT = 63
  CHART_COUNT = 3
  MAX_WINDOWS = 10000
  UPDATE_MARKET_DATA = False
  UPDATE_NEWS_DATA = False
# Models
  MODEL_PROFILE = 'online'   # "online" (Nano-evidence/debate, Mini-chart/supervisor), "default" (Gemma except Nano chart)
  OLLAMA_URL = 'http://127.0.0.1:11434'
# Decision
  DEBATE_MAX_ROUNDS = 2
  MEMORY_MODE = 'frozen_reference'   # "frozen_reference", "prequential"
  FEW_SHOT_MODE = 'frozen_reference'
  FEW_SHOT_MAX_EXAMPLES = 4
  FEW_SHOT_SCALER = 'none'
# Concurrency
  PIPELINE_MAX_IN_FLIGHT = 2
  MAX_CONCURRENT_MODEL_CALLS = 8
# Evaluation
  EVALUATION_RUN_ID = ''   # Optional completed run ID; leave empty when no run is selected
# Paths
  ## Results
     RUNS_PATH = 'Results/^GSPC'
     MEMORY_PATH = 'Results/^GS

In [3]:
def download_refinitiv_market(ric):
    for attempt in range(3):
        try:
            response = historical_pricing.summaries.Definition(universe=ric, interval=historical_pricing.Intervals.DAILY, start=download_start_date, end=end_date, fields=list(refinitiv_fields), adjustments="unadjusted").get_data()
            break
        except Exception as exc:
            transient = any(value in str(exc).lower() for value in ("timeout", "503", "502", "failed to connect"))
            if not transient or attempt == 2:
                raise
            time.sleep(2 ** attempt)
    raw = response.data.raw
    while isinstance(raw, list) and len(raw) == 1:
        raw = raw[0]
    if raw.get("universe", {}).get("ric") != ric:
        raise ValueError(f"Returned RIC does not match {ric}")
    frame = pd.DataFrame(raw.get("data", []), columns=[item["name"] for item in raw.get("headers", [])])
    if frame.empty or not {"DATE", "TRDPRC_1"}.issubset(frame.columns):
        raise ValueError(f"No dated closing prices returned for {ric}")
    frame["DATE"] = pd.to_datetime(frame["DATE"], errors="raise", utc=True).dt.tz_convert(None)
    frame = frame.set_index("DATE").sort_index()
    if frame.index.has_duplicates:
        raise ValueError(f"Duplicate dates returned for {ric}")
    frame = frame.loc[download_start_date:end_date].rename(columns=refinitiv_fields)
    frame = frame.reindex(columns=list(refinitiv_fields.values())).apply(pd.to_numeric, errors="coerce")
    if frame["Close"].dropna().le(0).any():
        raise ValueError(f"Nonpositive closing price returned for {ric}")
    frame.index.name = "Date"
    return frame, raw


def normalize_prices(frame):
    frame = frame.copy()
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = frame.columns.get_level_values(0)
    frame.index = pd.to_datetime(frame.index).tz_localize(None).normalize()
    if frame.index.has_duplicates:
        raise ValueError("Duplicate market dates.")
    frame = frame.reindex(columns=columns).apply(pd.to_numeric, errors="coerce").astype(float).sort_index()
    frame = frame.loc[download_start_date:end_date].replace([np.inf, -np.inf], np.nan)
    frame.index.name = "Date"
    return frame


def fill_prices(primary, fallback, fallback_name):
    dates = primary.index.union(fallback.index).sort_values().rename("Date")
    frame = primary.reindex(dates).copy()
    source = pd.DataFrame("", index=dates, columns=columns).mask(frame.notna(), "LSEG")
    fallback = fallback.reindex(dates)
    filled = frame.isna() & fallback.notna()
    frame = frame.fillna(fallback).astype(float)
    source = source.mask(filled, fallback_name)
    return frame, source


def interpolate_small_gaps(frame, source):
    frame, source = frame.copy(), source.copy()
    for column in columns:
        missing = frame[column].isna()
        gap_size = missing.groupby(missing.ne(missing.shift()).cumsum()).transform("sum")
        interpolated = frame[column].interpolate(method="linear", limit_area="inside")
        filled = missing & gap_size.le(max_interpolation_days) & interpolated.notna()
        frame.loc[filled, column] = interpolated.loc[filled]
        source.loc[filled, column] = "interpolation"
    return frame, source

In [4]:
refinitiv_data = {}
try:
    session = ld.open_session(name="desktop.workspace", app_key=os.getenv("LSEG_APP_KEY"))
    if session.open_state.name != "Opened":
        raise ConnectionError("Open and sign in to LSEG Workspace on this computer.")
    for asset, ric in refinitiv_assets.items():
        frame, raw = download_refinitiv_market(ric)
        folder = Path("Data") / asset / "Refinitiv"
        folder.mkdir(parents=True, exist_ok=True)
        (folder / "historical_pricing.json").write_text(json.dumps(raw, ensure_ascii=False, indent=2), encoding="utf-8")
        if asset == "CBU0.L":
            eod = ld.get_data(ric, ["TR.PriceClose.date", "TR.PriceClose", "TR.PriceOpen", "TR.PriceHigh", "TR.PriceLow", "TR.Volume"], parameters={"SDate": download_start_date, "EDate": end_date, "Frq": "D", "Adjusted": 0})
            eod.to_csv(folder / "eod_prices.csv", index=False)
            if eod.empty or eod["Date"].isna().any():
                raise ValueError("CBU0.L: no dated exchange closing prices returned.")
            frame = normalize_prices(eod.set_index("Date").rename(columns={"Price Close": "Close", "Price Open": "Open", "Price High": "High", "Price Low": "Low"}))
        frame.to_csv(folder / "df_OHLCV.csv")
        metadata = {"asset": asset, "ric": ric, "requested_start": download_start_date, "requested_end": end_date, "rows": len(frame), "start": str(frame.index.min().date()), "end": str(frame.index.max().date()), "missing_values": frame.isna().sum().to_dict(), "adjustments": "unadjusted", "price_source": "LSEG exchange EOD (TR.Price fields)" if asset == "CBU0.L" else "LSEG Historical Pricing", "retrieved_at_utc": pd.Timestamp.now(tz="UTC").isoformat()}
        (folder / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
        refinitiv_data[asset] = normalize_prices(frame)
        print(f"{asset}: {len(frame)} LSEG rows, {int(frame.isna().sum().sum())} missing values")
finally:
    ld.close_session()

^GSPC: 894 LSEG rows, 894 missing values
SPGTCLTR: 930 LSEG rows, 3720 missing values


STOXX: 913 LSEG rows, 0 missing values


CACT: 909 LSEG rows, 3 missing values


GDAXI: 905 LSEG rows, 4 missing values


CBU0.L: 900 LSEG rows, 0 missing values


FTSE100: 900 LSEG rows, 0 missing values
DJCI: 896 LSEG rows, 896 missing values


In [5]:
yahoo_data = {}
yahoo_errors = {}
for asset, symbol in yahoo_assets.items():
    folder = Path("Data") / asset / "yfinance"
    folder.mkdir(parents=True, exist_ok=True)
    yahoo_data[asset] = pd.DataFrame(columns=columns, index=pd.DatetimeIndex([], name="Date"))
    if symbol is None:
        yahoo_errors[asset] = "No verified Yahoo equivalent configured."
        continue
    try:
        raw = yf.download(symbol, start=download_start_date, end=(pd.Timestamp(end_date) + pd.Timedelta(days=1)).strftime("%Y-%m-%d"), auto_adjust=False, progress=False)
        if raw.empty:
            raise ValueError("Yahoo returned no data.")
        raw.to_csv(folder / "download.csv")
        yahoo_data[asset] = normalize_prices(raw).dropna(how="all")
        yahoo_data[asset].to_csv(folder / "df_OHLCV.csv")
    except Exception as exc:
        yahoo_errors[asset] = f"{type(exc).__name__}: {exc}"
        print(f"{asset}: {yahoo_errors[asset]}")

$^SBF250: possibly delisted; no price data found  (1d 2023-02-07 -> 2026-09-01)



1 Failed download:


['^SBF250']: possibly delisted; no price data found  (1d 2023-02-07 -> 2026-09-01)


CACT: ValueError: Yahoo returned no data.


$^DJCI: possibly delisted; no price data found  (1d 2023-02-07 -> 2026-09-01)



1 Failed download:


['^DJCI']: possibly delisted; no price data found  (1d 2023-02-07 -> 2026-09-01)


DJCI: ValueError: Yahoo returned no data.


In [6]:
market_data = {}
fill_sources = {}
summary = []
for asset, primary in refinitiv_data.items():
    # Each asset uses only dates reported by its own sources, never a shared business-day grid.
    frame, source = fill_prices(primary, yahoo_data[asset], "yfinance")
    if asset in other_source_files and frame.isna().any().any():
        other = normalize_prices(pd.read_csv(other_source_files[asset], index_col="Date", parse_dates=["Date"]))
        other = other.dropna(how="all").reindex(frame.index)
        filled = frame.isna() & other.notna()
        frame = frame.fillna(other)
        source = source.mask(filled, str(other_source_files[asset]))
    frame, source = interpolate_small_gaps(frame, source)
    # Interpolation uses a later observation; its cells remain explicitly identified for research use.
    if frame[columns[:-1]].le(0).any().any() or frame["Volume"].lt(0).any():
        raise ValueError(f"{asset}: nonpositive prices or negative volume.")
    frame["return"] = np.log(frame["Close"]).diff()
    market_data[asset], fill_sources[asset] = frame, source
    missing = int(frame[columns].isna().sum().sum())
    summary.append({"asset": asset, "start": frame.index.min().date(), "end": frame.index.max().date(), "rows": len(frame), "fallback_cells": int(((source != "") & (source != "LSEG") & (source != "interpolation")).sum().sum()), "interpolated_cells": int(source.eq("interpolation").sum().sum()), "missing_values": missing, "status": "complete" if missing == 0 else "incomplete", "yahoo_note": yahoo_errors.get(asset, "")})
display(pd.DataFrame(summary).set_index("asset"))

,start,end,rows,fallback_cells,interpolated_cells,missing_values,status,yahoo_note
asset,,,,,,,,
^GSPC,2023-02-07,2026-08-31,894,894,0,0,complete,
SPGTCLTR,2023-02-07,2026-08-31,930,0,0,3720,incomplete,No verified Yahoo equivalent configured.
STOXX,2023-02-07,2026-08-31,913,0,0,0,complete,
CACT,2023-02-07,2026-08-31,909,0,3,0,complete,ValueError: Yahoo returned no data.
GDAXI,2023-02-07,2026-08-31,905,4,0,0,complete,
CBU0.L,2023-02-07,2026-08-28,900,0,0,0,complete,
FTSE100,2023-02-07,2026-08-28,900,0,0,0,complete,
DJCI,2023-02-07,2026-08-31,896,0,0,896,incomplete,ValueError: Yahoo returned no data.


In [7]:
for result in summary:
    asset = result["asset"]
    folder = Path("Data") / asset / "Market"
    folder.mkdir(parents=True, exist_ok=True)
    frame = market_data[asset]
    exported = frame.iloc[1:]
    exported.to_csv(folder / "df_OHLCV.csv")
    fill_sources[asset].to_csv(folder / "fill_sources.csv")
    metadata = {**result, "rows": len(exported), "start": str(exported.index.min().date()), "end": str(exported.index.max().date()), "missing_values": int(exported[columns].isna().sum().sum()), "analysis_start": analysis_start_date, "requested_start": download_start_date, "requested_end": end_date, "ric": refinitiv_assets[asset], "yahoo_symbol": yahoo_assets[asset], "max_consecutive_interpolation_days": max_interpolation_days, "interpolation_uses_future_observation": result["interpolated_cells"] > 0, "return_definition": "log(Close).diff(); first observation omitted", "retrieved_at_utc": pd.Timestamp.now(tz="UTC").isoformat()}
    (folder / "metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
print("Saved Refinitiv, yfinance and processed Market files. Incomplete assets retain missing values.")

Saved Refinitiv, yfinance and processed Market files. Incomplete assets retain missing values.
